# Indonesia Provincial GVA — 2020–2026

**Inputs:**
- `iot_2020_gva_ratios.csv` — sectoral GVA multipliers from the 2020 Input–Output Table
- `02_01_pdrb_sectoral_long.csv` — provincial PDRB by sector, both price bases (output of `02_01` notebook)

**Output:** `03_01_provincial_gva.csv` — provincial GVA by sector, price basis, and period with YoY growth rates and sector groupings

#### Key methodological notes

1. **GVA multiplier is applied sector-by-sector.**  
   `GVA_provincial[sector] = PDRB_provincial[sector] × gva_multiplier[sector]`  
   where `gva_multiplier = gva_basic / gdp_market` derived from 2020 IOT rows 2090 and 1950.

2. **Computed for both ADHB and ADHK; ADHK is the primary display basis.**  
   ADHK (constant 2010 prices) is used for all growth rates and YoY comparisons — it strips price effects.  
   ADHB (current prices) is retained for structural share analysis (GVA as % of GDRP) and is always available in the background.

3. **Frozen coefficient assumption.** 2020 IOT multipliers are held constant across all 38 provinces and all years. Provincial tax structures are assumed to equal the national average by sector.

4. **Pertanian multiplier exceeds 1.** Agriculture receives net product subsidies (row 1950 < 0), so GVA at basic prices exceeds PDRB at market prices for this sector. This is correct per SNA 2008.

5. **New Papua provinces excluded for 2020–2022.** Papua Barat Daya, Papua Selatan, Papua Tengah, and Papua Pegunungan have no BPS sectoral PDRB before 2023. The full time-series uses 34 provinces; 38 provinces are available from 2023 onward.


## 1. Packages setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import ipywidgets as widgets
    from ipywidgets import interact
    widget_support = True
except ImportError:
    widget_support = False

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:,.4f}'.format)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f'Widget support: {widget_support}')


Widget support: True


## 2. Data paths

In [2]:
\
from pathlib import Path

BASE   = Path(r'C:\Users\Admin\OneDrive\Desktop\Personal Projects\Indonesia GVA')
RAW    = BASE / '00_base_data'
INTER  = BASE / '02_intermediate_data'
OUT    = BASE / '02_intermediate_data'

# PDRB panel — both price bases in one file (output of 02_01 notebook)
pdrb_long = pd.read_csv(INTER / '02_01_pdrb_sectoral_long.csv')

# IOT GVA multipliers
gva_ratios = pd.read_csv(INTER / '02_01_iot_2020_gva_ratios.csv')

print('PDRB rows :', len(pdrb_long))
print('Price bases:', pdrb_long['price_basis'].value_counts().to_dict())
print('\nGVA ratios shape:', gva_ratios.shape)
print(gva_ratios.head())


PDRB rows : 34200
Price bases: {'ADHB': 17100, 'ADHK': 17100}

GVA ratios shape: (17, 13)
                                       industry_name      labour_comp  \
0                Pertanian, Kehutanan, dan Perikanan 758,855,037.0000   
1                        Pertambangan dan Penggalian 224,600,593.0000   
2                                Industri Pengolahan 975,126,797.0000   
3                          Pengadaan Listrik dan Gas  33,575,863.0000   
4  Pengadaan Air, Pengelolaan Sampah, Limbah dan ...   9,973,317.0000   

       surplus_usaha  other_taxes_prod          gva_basic  net_taxes_product  \
0 1,282,990,498.0000    9,009,517.0000 2,050,855,052.0000   -26,795,579.0000   
1   749,888,356.0000    5,354,698.0000   979,843,647.0000     4,767,425.0000   
2 2,136,615,725.0000   30,050,794.0000 3,141,793,316.0000    56,196,711.0000   
3   126,444,722.0000      354,163.0000   160,374,748.0000     1,136,121.0000   
4    33,610,772.0000       92,686.0000    43,676,775.0000       235,787

## 3. GVA calculation

Apply the IOT multiplier to every province × sector × period combination,  
**separately for ADHB and ADHK**.  All downstream figures use ADHK;  
ADHB is kept as a background column for structural analysis.


In [3]:
print("pdrb_long sector_names:")
print(sorted(pdrb_long['sector_name'].unique()))

print("\ngva_ratios industry_names:")
print(sorted(gva_ratios['industry_name'].unique()))

pdrb_long sector_names:
['Administrasi Pemerintahan, Pertahanan dan Jaminan Sosial Wajib', 'Industri Pengolahan', 'Informasi dan Komunikasi', 'Jasa Kesehatan dan Kegiatan Sosial', 'Jasa Keuangan dan Asuransi', 'Jasa Lainnya', 'Jasa Pendidikan', 'Jasa Perusahaan', 'Konstruksi', 'Pengadaan Air, Pengelolaan Sampah, Limbah dan Daur Ulang', 'Pengadaan Listrik dan Gas', 'Penyediaan Akomodasi dan Makan Minum', 'Perdagangan Besar dan Eceran, Reparasi Mobil dan Sepeda Motor', 'Pertambangan dan Penggalian', 'Pertanian, Kehutanan dan Perikanan', 'Produk Domestik Regional Bruto', 'Real Estate', 'Transportasi dan Pergudangan']

gva_ratios industry_names:
['Administrasi Pemerintahan, Pertahanan dan Jaminan Sosial Wajib', 'Industri Pengolahan', 'Informasi dan Komunikasi', 'Jasa Kesehatan dan Kegiatan Sosial', 'Jasa Keuangan dan Asuransi', 'Jasa Pendidikan', 'Jasa Perusahaan', 'Jasa lainnya', 'Konstruksi', 'Pengadaan Air, Pengelolaan Sampah, Limbah dan Daur Ulang', 'Pengadaan Listrik dan Gas', 'Penyed

In [4]:
NAME_FIX = {
    'Pertanian, Kehutanan, dan Perikanan'                          : 'Pertanian, Kehutanan dan Perikanan',
    'Perdagangan Besar dan Eceran; Reparasi Mobil dan Sepeda Motor': 'Perdagangan Besar dan Eceran, Reparasi Mobil dan Sepeda Motor',
    'Jasa lainnya'                                                 : 'Jasa Lainnya',
}

gva_ratios['industry_name'] = gva_ratios['industry_name'].replace(NAME_FIX)

df = pdrb_long.merge(
    gva_ratios[['industry_name', 'gva_share_of_gdp']],
    left_on='sector_name',
    right_on='industry_name',
    how='left'
).drop(columns='industry_name')

missing_mult = df['gva_share_of_gdp'].isna().sum()
if missing_mult > 0:
    print(f'WARNING: {missing_mult} rows with no multiplier (sectors):',
          df.loc[df['gva_share_of_gdp'].isna(), 'sector_code'].unique())

df['gva_sector'] = df['value_billion_idr'] * df['gva_share_of_gdp']

print('Rows in merged dataset:', len(df))
print(df[['price_basis','provinsi','period','sector_code',
          'value_billion_idr','gva_share_of_gdp','gva_sector']].head(6).to_string(index=False))

Rows in merged dataset: 34200
price_basis provinsi period sector_code  value_billion_idr  gva_share_of_gdp  gva_sector
       ADHB     Aceh 2020Q1           A        13,233.9000            1.0132 13,409.0974
       ADHB     Aceh 2020Q1           B         1,723.6200            0.9952  1,715.2743
       ADHB     Aceh 2020Q1           C         1,727.1700            0.9824  1,696.8193
       ADHB     Aceh 2020Q1           D            56.5100            0.9930     56.1125
       ADHB     Aceh 2020Q1           E            19.5800            0.9946     19.4749
       ADHB     Aceh 2020Q1           F         4,259.6600            0.9781  4,166.3954


In [5]:
# ── Split into ADHK (primary) and ADHB (background) ─────────────────────────
df_adhk = df[df['price_basis'] == 'ADHK'].copy().reset_index(drop=True)
df_adhb = df[df['price_basis'] == 'ADHB'].copy().reset_index(drop=True)

print(f'ADHK rows: {len(df_adhk):,}  |  ADHB rows: {len(df_adhb):,}')
print('ADHK periods:', sorted(df_adhk['period'].unique()))
print('ADHB periods:', sorted(df_adhb['period'].unique()))


ADHK rows: 17,100  |  ADHB rows: 17,100
ADHK periods: ['2020Q1', '2020Q2', '2020Q3', '2020Q4', '2021Q1', '2021Q2', '2021Q3', '2021Q4', '2022Q1', '2022Q2', '2022Q3', '2022Q4', '2023Q1', '2023Q2', '2023Q3', '2023Q4', '2024Q1', '2024Q2', '2024Q3', '2024Q4', '2025Q1', '2025Q2', '2025Q3', '2025Q4', '2026Q1']
ADHB periods: ['2020Q1', '2020Q2', '2020Q3', '2020Q4', '2021Q1', '2021Q2', '2021Q3', '2021Q4', '2022Q1', '2022Q2', '2022Q3', '2022Q4', '2023Q1', '2023Q2', '2023Q3', '2023Q4', '2024Q1', '2024Q2', '2024Q3', '2024Q4', '2025Q1', '2025Q2', '2025Q3', '2025Q4', '2026Q1']


In [6]:
# ── Sector short-labels (used throughout visualisations) ─────────────────────
SECTOR_SHORT = {
    'Pertanian, Kehutanan dan Perikanan'                                    : 'Pertanian',
    'Pertambangan dan Penggalian'                                           : 'Pertambangan',
    'Industri Pengolahan'                                                   : 'Industri Pengolahan',
    'Pengadaan Listrik dan Gas'                                             : 'Listrik & Gas',
    'Pengadaan Air, Pengelolaan Sampah, Limbah dan Daur Ulang'             : 'Air & Sampah',
    'Konstruksi'                                                            : 'Konstruksi',
    'Perdagangan Besar dan Eceran, Reparasi Mobil dan Sepeda Motor'        : 'Perdagangan',
    'Transportasi dan Pergudangan'                                          : 'Transportasi',
    'Penyediaan Akomodasi dan Makan Minum'                                 : 'Akomodasi',
    'Informasi dan Komunikasi'                                              : 'Infokom',
    'Jasa Keuangan dan Asuransi'                                           : 'Keuangan',
    'Real Estate'                                                           : 'Real Estate',
    'Jasa Perusahaan'                                                       : 'Jasa Perusahaan',
    'Administrasi Pemerintahan, Pertahanan dan Jaminan Sosial Wajib'       : 'Adm. Pemerintahan',
    'Jasa Pendidikan'                                                       : 'Pendidikan',
    'Jasa Kesehatan dan Kegiatan Sosial'                                   : 'Kesehatan',
    'Jasa Lainnya'                                                          : 'Jasa Lainnya',
    'Produk Domestik Regional Bruto'                                        : 'PDRB Total',
}

# ── Sector groupings ─────────────────────────────────────────────────────────
MARKET_SECTORS = {'A','B','C','D','E','F','G','H','I','J','K','L','MN'}
GOVT_SECTORS   = {'O','P','Q','RSTU'}

def tag_groups(d):
    d['market_sector'] = d['sector_code'].isin(MARKET_SECTORS)
    d['govt_proximate'] = d['sector_code'].isin(GOVT_SECTORS)
    return d

df_adhk = tag_groups(df_adhk)
df_adhb = tag_groups(df_adhb)
print('Sector grouping applied.')


Sector grouping applied.


## 4. YoY growth rates (ADHK only)

Growth rates are always computed on ADHK to strip price effects.  
Annual YoY: `(year_t / year_{t-1}) - 1`, using the **annual (Tahunan)** figure  
— or quarterly if annual is unavailable.


In [7]:
# ── Annual aggregates (use 'Tahunan' if present, else sum Q1-Q4) ─────────────
# The 02_01 notebook already drops Tahunan — so we sum Q1-Q4 per year.

def annual_sum(d):
    return (
        d[d['sector_code'] != 'PDRB']           # exclude the pre-aggregated total row
        .groupby(['price_basis','provinsi','year','sector_code','sector_name',
                  'market_sector','govt_proximate'], dropna=False)
        ['gva_sector']
        .sum(min_count=1)                        # require at least 1 quarter, else NaN
        .reset_index()
    )

ann_adhk = annual_sum(df_adhk)
ann_adhb = annual_sum(df_adhb)

print(f'Annual ADHK rows: {len(ann_adhk):,}')
print(ann_adhk.head(3).to_string(index=False))


Annual ADHK rows: 4,522
price_basis provinsi  year sector_code                        sector_name  market_sector  govt_proximate  gva_sector
       ADHK     Aceh  2020           A Pertanian, Kehutanan dan Perikanan           True           False 38,401.2541
       ADHK     Aceh  2020           B        Pertambangan dan Penggalian           True           False 10,434.5607
       ADHK     Aceh  2020           C                Industri Pengolahan           True           False  5,952.1843


In [8]:
# ── Quarterly YoY (ADHK) — same quarter, prior year ─────────────────────────
# This is the correct approach: 2026Q1 vs 2025Q1, 2025Q4 vs 2024Q4, etc.

qtr_adhk = (
    df_adhk[df_adhk['sector_code'] != 'PDRB']
    .sort_values(['provinsi', 'sector_code', 'year', 'quarter'])
    .copy()
)

qtr_adhk['gva_sector'] = qtr_adhk['value_billion_idr'] * qtr_adhk['sector_name'].map(
    gva_ratios.set_index('industry_name')['gva_share_of_gdp']
)

qtr_adhk['gva_yoy'] = (
    qtr_adhk
    .groupby(['provinsi', 'sector_code', 'quarter'])['gva_sector']
    .pct_change(fill_method=None) * 100
)
qtr_adhk['sector_short'] = qtr_adhk['sector_name'].map(SECTOR_SHORT)

# ── Provincial totals (ADHK, quarterly) ──────────────────────────────────────
qtr_adhk['gva_market'] = qtr_adhk['gva_sector'].where(qtr_adhk['market_sector'], 0)
qtr_adhk['gva_govt']   = qtr_adhk['gva_sector'].where(qtr_adhk['govt_proximate'], 0)

prov_adhk = (
    qtr_adhk
    .groupby(['provinsi', 'year', 'quarter', 'period'])
    .agg(
        gva_total          = ('gva_sector', 'sum'),
        market_gva         = ('gva_market',  'sum'),
        govt_proximate_gva = ('gva_govt',    'sum'),
    )
    .reset_index()
    .sort_values(['provinsi', 'year', 'quarter'])
)
prov_adhk['gva_total_yoy'] = (
    prov_adhk
    .groupby(['provinsi', 'quarter'])['gva_total']
    .pct_change(fill_method=None) * 100
)
prov_adhk['market_share_pct']         = prov_adhk['market_gva']         / prov_adhk['gva_total'] * 100
prov_adhk['govt_proximate_share_pct'] = prov_adhk['govt_proximate_gva'] / prov_adhk['gva_total'] * 100

print('Provincial ADHK totals sample (latest period):')
print(prov_adhk[prov_adhk['period'] == prov_adhk['period'].max()]
      .head(5).to_string(index=False))

Provincial ADHK totals sample (latest period):
     provinsi  year quarter period    gva_total   market_gva  govt_proximate_gva  gva_total_yoy  market_share_pct  govt_proximate_share_pct
         Aceh  2026      Q1 2026Q1  39,847.6318  33,400.3351          6,447.2967         4.0584           83.8201                   16.1799
         Bali  2026      Q1 2026Q1  43,998.7190  36,716.1689          7,282.5501         5.5701           83.4483                   16.5517
       Banten  2026      Q1 2026Q1 143,548.9725 132,553.3969         10,995.5756         5.6622           92.3402                    7.6598
     Bengkulu  2026      Q1 2026Q1  14,272.8743  11,604.1281          2,668.7462         4.7183           81.3020                   18.6980
DI Yogyakarta  2026      Q1 2026Q1  33,904.3568  26,572.8400          7,331.5168         5.8284           78.3759                   21.6241


In [9]:
# ── PDRB benchmarks (ADHK, quarterly YoY) ────────────────────────────────────
pdrb_prov_yoy = (
    df_adhk[df_adhk['sector_code'] == 'PDRB']
    .groupby(['provinsi', 'year', 'quarter', 'period'])['value_billion_idr']
    .sum()
    .reset_index()
    .sort_values(['provinsi', 'year', 'quarter'])
)
pdrb_prov_yoy['pdrb_total_yoy'] = (
    pdrb_prov_yoy
    .groupby(['provinsi', 'quarter'])['value_billion_idr']
    .pct_change(fill_method=None) * 100
)

pdrb_sector_yoy = (
    df_adhk[df_adhk['sector_code'] != 'PDRB']
    .sort_values(['provinsi', 'sector_code', 'year', 'quarter'])
    .copy()
)
pdrb_sector_yoy['pdrb_sector_yoy'] = (
    pdrb_sector_yoy
    .groupby(['provinsi', 'sector_code', 'quarter'])['value_billion_idr']
    .pct_change(fill_method=None) * 100
)
pdrb_sector_yoy['sector_short'] = pdrb_sector_yoy['sector_name'].map(SECTOR_SHORT)

province_list = sorted(qtr_adhk['provinsi'].dropna().unique())
print(f'Provinces available: {len(province_list)}')
print('Latest periods:', sorted(pdrb_prov_yoy['period'].unique())[-4:])

Provinces available: 38
Latest periods: ['2025Q2', '2025Q3', '2025Q4', '2026Q1']


## 5. Structural shares (ADHB background)

GVA as a share of GDRP uses **ADHB** (current prices) — the correct basis  
for structural composition analysis. ADHK shares would embed base-year  
distortions from relative price shifts since 2010.


In [12]:
# Annual ADHB aggregates (same annual-sum logic)
ann_adhb_sorted = ann_adhb.sort_values(['provinsi','sector_code','year'])
ann_adhb_sorted['sector_short'] = ann_adhb_sorted['sector_name'].map(SECTOR_SHORT)

# PDRB market-price total by province-year (ADHB) — denominator for GVA/GDP ratio
pdrb_adhb_total = (
    df_adhb[df_adhb['sector_code'] == 'PDRB']
    .groupby(['provinsi','year'])['value_billion_idr']
    .sum()
    .rename('pdrb_market_total')
    .reset_index()
)

ann_adhb_sorted = ann_adhb_sorted.merge(pdrb_adhb_total, on=['provinsi','year'], how='left')
ann_adhb_sorted['gva_pct_gdp'] = ann_adhb_sorted['gva_sector'] / ann_adhb_sorted['pdrb_market_total'] * 100

print('ADHB structural table sample:')
print(ann_adhb_sorted[ann_adhb_sorted['year']==ann_adhb_sorted['year'].max()]
      [['provinsi','sector_short','year','gva_sector','pdrb_market_total','gva_pct_gdp']]
      .head(6).to_string(index=False))


ADHB structural table sample:
provinsi        sector_short  year  gva_sector  pdrb_market_total  gva_pct_gdp
    Aceh           Pertanian  2026 21,324.0913        66,390.9200      32.1190
    Aceh        Pertambangan  2026  4,045.4270        66,390.9200       6.0933
    Aceh Industri Pengolahan  2026  3,131.3992        66,390.9200       4.7166
    Aceh       Listrik & Gas  2026     68.9019        66,390.9200       0.1038
    Aceh        Air & Sampah  2026     30.5252        66,390.9200       0.0460
    Aceh          Konstruksi  2026  5,663.7570        66,390.9200       8.5309


## 6. Export

In [15]:
# ── Re-derive annual ADHK by summing Q1–Q4 ───────────────────────────────────
# Require all 4 quarters; 2026 (Q1 only) will be NaN and excluded
ann_adhk_sorted = (
    qtr_adhk[qtr_adhk['sector_code'] != 'PDRB']
    .groupby(['provinsi', 'year', 'sector_code', 'sector_name', 'sector_short',
              'market_sector', 'govt_proximate'], dropna=False)
    ['gva_sector']
    .sum(min_count=4)
    .reset_index()
    .sort_values(['provinsi', 'sector_code', 'year'])
)
ann_adhk_sorted['gva_yoy'] = (
    ann_adhk_sorted
    .groupby(['provinsi', 'sector_code'])['gva_sector']
    .pct_change(fill_method=None) * 100
)

print(f'Annual ADHK rows: {len(ann_adhk_sorted):,}')
print('Years:', sorted(ann_adhk_sorted['year'].dropna().unique()))

# ── Annual export (ADHK + ADHB) ───────────────────────────────────────────────
export_adhk_ann = ann_adhk_sorted[
    ['provinsi', 'year', 'sector_code', 'sector_name', 'sector_short',
     'market_sector', 'govt_proximate', 'gva_sector', 'gva_yoy']
].assign(price_basis='ADHK')

export_adhb_ann = ann_adhb_sorted[
    ['provinsi', 'year', 'sector_code', 'sector_name', 'sector_short',
     'market_sector', 'govt_proximate', 'gva_sector', 'gva_pct_gdp']
].assign(price_basis='ADHB')

combined_ann = pd.concat([export_adhk_ann, export_adhb_ann], ignore_index=True)
combined_ann = combined_ann.sort_values(['price_basis', 'provinsi', 'year', 'sector_code'])

out_ann = OUT / '03_01_provincial_gva_annual.csv'
combined_ann.to_csv(out_ann, index=False)
print(f'Annual saved    -> {out_ann}  ({len(combined_ann):,} rows)')

# ── Quarterly export (ADHK only) ──────────────────────────────────────────────
export_adhk_qtr = qtr_adhk[
    ['provinsi', 'year', 'quarter', 'period', 'sector_code', 'sector_name', 'sector_short',
     'market_sector', 'govt_proximate', 'gva_sector', 'gva_yoy']
].assign(price_basis='ADHK')

out_qtr = OUT / '03_01_provincial_gva_quarterly.csv'
export_adhk_qtr.to_csv(out_qtr, index=False)
print(f'Quarterly saved -> {out_qtr}  ({len(export_adhk_qtr):,} rows)')

Annual ADHK rows: 4,522
Years: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
Annual saved    -> C:\Users\Admin\OneDrive\Desktop\Personal Projects\Indonesia GVA\02_intermediate_data\03_01_provincial_gva_annual.csv  (9,044 rows)
Quarterly saved -> C:\Users\Admin\OneDrive\Desktop\Personal Projects\Indonesia GVA\02_intermediate_data\03_01_provincial_gva_quarterly.csv  (16,150 rows)
